<a href="https://colab.research.google.com/github/mcallem/Clase_Git/blob/main/Copia_de_10_2_Sesgos_Dataset_Adult.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Tarea: ¿Son justos los datos? Analizando sesgos en el Adult Income Dataset


**Módulo II: Ética en IA — Sesgos en datasets**

---

## 🎯 ¿Qué vas a hacer en esta tarea?

En clase vimos cómo Amazon entrenó una IA con datos históricos sesgados... y la IA aprendió a discriminar.

En esta tarea vas a hacer exactamente lo que debería haber hecho el equipo de Amazon **antes** de entrenar: **auditar el dataset**.

Vas a explorar el **Adult Income Dataset**, un conjunto de datos del censo de EE.UU. de 1994, y responder a una pregunta clave:

> ### ❓ ¿El dataset trata igual a hombres y mujeres? ¿Y a personas de distintos orígenes?

---

## 📋 Instrucciones generales

- Las celdas con `# 👉 TU CÓDIGO AQUÍ` son las que debes completar.
- El resto del código ya está escrito — léelo, ejecútalo y entiéndelo.
- Al final hay **preguntas de reflexión** que debes responder en texto.
- **No hay respuestas incorrectas en la reflexión** — lo importante es que razonas.

---

## 📦 Sobre el dataset

El **Adult Income Dataset** (también llamado "Census Income") contiene datos de ~48.000 personas de EE.UU. en 1994. Cada fila es una persona, y la columna objetivo es:

- `income`: si esa persona gana **más de 50.000$ al año** (`>50K`) o **menos** (`<=50K`)

Las columnas incluyen: edad, nivel educativo, ocupación, horas de trabajo, género, etnia, país de origen...

> 💭 **Antes de empezar, piensa:** Si una IA aprende de este dataset para predecir quién merece un crédito o un ascenso, ¿qué podría salir mal?

---
## ⚙️ Parte 0 — Preparación: importar librerías y cargar datos

In [ ]:
# Librerías necesarias — ejecuta esta celda primero
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Paleta de colores
COLOR_PURPLE = '#4211BA'
COLOR_TEAL   = '#33C6CD'
COLOR_ORANGE = '#F97316'
COLOR_RED    = '#EF4444'
COLOR_GRAY   = '#94A3B8'

print('✅ Librerías cargadas correctamente')

In [ ]:
columnas = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race',
    'sex', 'capital_gain', 'capital_loss', 'hours_per_week',
    'native_country', 'income'
]

# Una sola lectura del archivo subido
df = pd.read_csv('adult.csv', names=columnas, skipinitialspace=True, header=0)

# Limpieza
df = df[df['occupation'] != '?']
df = df[df['workclass'] != '?']
df = df[df['native_country'] != '?']
df = df.reset_index(drop=True)

# Conversión de tipos
cols_numericas = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
df[cols_numericas] = df[cols_numericas].apply(pd.to_numeric, errors='coerce')

print(f'✅ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas')
print(df.dtypes)

---
## 🔎 Parte 1 — Primera exploración: ¿Qué hay en el dataset?

Antes de buscar sesgos, necesitamos conocer los datos. Esto se llama **EDA (Exploratory Data Analysis)**.

In [ ]:
# Veamos las primeras filas
df.head()

In [ ]:
# Información general del dataframe
df.info()

In [ ]:
# Estadísticas descriptivas de las columnas numéricas
df.describe()

### 📝 Ejercicio 1.1 — ¿Cuántas personas hay en el dataset?

Completa el código para mostrar:
- El número total de personas en el dataset
- Cuántas ganan `>50K` y cuántas `<=50K`

In [ ]:
# Total de personas en el dataset
total = len(df)
print(f'Total de personas en el dataset: {total}')

# 👉 TU CÓDIGO AQUÍ
# Usa df['income'].value_counts() para ver cuántas personas hay en cada categoría de ingresos
conteo_ingresos = _______________
print('\nDistribución de ingresos:')
print(conteo_ingresos)

### 📝 Ejercicio 1.2 — ¿Cómo se distribuye el género en el dataset?

Vamos a ver cuántos hombres y mujeres hay, y qué porcentaje representan.

In [ ]:
# Distribución de género
conteo_genero = df['sex'].value_counts()
porcentaje_genero = df['sex'].value_counts(normalize=True) * 100

print('Distribución de género:')
print(conteo_genero)
print('\nEn porcentaje:')

# 👉 TU CÓDIGO AQUÍ
# Imprime porcentaje_genero redondeando a 1 decimal
# Pista: usa .round(1)
print(_______________)

# Visualización — ya está hecho, solo ejecútalo
fig, ax = plt.subplots()
bars = ax.bar(conteo_genero.index, conteo_genero.values,
              color=[COLOR_PURPLE, COLOR_TEAL], width=0.5)
for bar, val in zip(bars, conteo_genero.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', fontsize=11, fontweight='bold')
ax.set_title('¿Cuántos hombres y mujeres hay en el dataset?', fontsize=13, pad=15)
ax.set_ylabel('Número de personas')
ax.set_xlabel('Género')
plt.tight_layout()
plt.show()

print('\n💬 ¿Ya ves algo sospechoso? Anótalo.')

---
## 🚨 Parte 2 — Análisis de sesgo por género

> **Pregunta clave:** ¿Qué porcentaje de hombres gana >50K vs. qué porcentaje de mujeres?

Si los porcentajes son muy distintos, tenemos una señal de sesgo de género en los datos.

### 📝 Ejercicio 2.1 — Porcentaje de ingresos altos por género

In [ ]:
# Calculamos el porcentaje de personas con >50K por género
# groupby agrupa por género, luego calculamos la media de si income es >50K

# Primero creamos una columna binaria: 1 si gana >50K, 0 si no
df['income_alto'] = (df['income'] == '>50K').astype(int)

# 👉 TU CÓDIGO AQUÍ
# Agrupa df por 'sex' y calcula la media de 'income_alto'
# Multiplica por 100 para obtener el porcentaje
# Pista: df.groupby('sex')['income_alto'].mean() * 100
pct_por_genero = _______________

print('Porcentaje de personas que ganan >50K, por género:')
print(pct_por_genero.round(1))

In [ ]:
# Visualización del resultado
fig, ax = plt.subplots()
colores = [COLOR_PURPLE, COLOR_TEAL]
bars = ax.bar(pct_por_genero.index, pct_por_genero.values, color=colores, width=0.4)

for bar, val in zip(bars, pct_por_genero.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontsize=12, fontweight='bold')

ax.set_title('¿Qué porcentaje gana >50K, según género?', fontsize=13, pad=15)
ax.set_ylabel('% que gana >50K')
ax.set_ylim(0, 45)
ax.axhline(y=df['income_alto'].mean()*100, color=COLOR_GRAY, linestyle='--',
           linewidth=1.2, label=f'Media global ({df["income_alto"].mean()*100:.1f}%)')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Ejercicio 2.2 — ¿Cuántas veces más probable es que un hombre gane >50K que una mujer?

In [ ]:
pct_hombre = pct_por_genero['Male']
pct_mujer  = pct_por_genero['Female']

# 👉 TU CÓDIGO AQUÍ
# Calcula cuántas veces mayor es pct_hombre respecto a pct_mujer

ratio = _______________

print(f'Un hombre tiene {ratio:.1f}x más probabilidades de ganar >50K que una mujer.')
print(f'\nHombres con >50K: {pct_hombre:.1f}%')
print(f'Mujeres con >50K:  {pct_mujer:.1f}%')

### 📝 Ejercicio 2.3 — ¿Es solo por las horas de trabajo?

Una explicación posible es que los hombres trabajen más horas. Vamos a comprobarlo.

In [ ]:
# Media de horas semanales por género
horas_genero = df.groupby('sex')['hours_per_week'].mean()
print('Media de horas semanales por género:')
print(horas_genero.round(1))

# 👉 TU CÓDIGO AQUÍ
# Calcula también la media de años de educación (columna 'education_num') por género
# Pista: igual que el código de arriba pero con 'education_num'
educ_genero = _______________
print('\nMedia de años de educación por género:')
print(educ_genero.round(1))

print('\n💬 ¿Justifican las horas de trabajo o la educación la diferencia de ingresos?')

---
## 🚨 Parte 3 — Análisis de sesgo por etnia

> **Pregunta clave:** ¿Varía la proporción de ingresos altos según la etnia?

Primero, veamos qué tipo de población o etnia hay en el dataset.

In [ ]:
# Distribución de etnias en el dataset
conteo_etnia = df['race'].value_counts()
print('Distribución de etnias en el dataset:')
print(conteo_etnia)
print(f'\n⚠️  ¿Ves algún problema de representación?')

### 📝 Ejercicio 3.1 — Porcentaje de ingresos altos por etnias

In [ ]:
# 👉 TU CÓDIGO AQUÍ
# Calcula el porcentaje de personas con >50K por etnias
# Es igual que en el ejercicio 2.1 pero agrupando por 'race'
pct_por_etnia = _______________
pct_por_etnia = pct_por_etnia.sort_values(ascending=False)

print('Porcentaje de personas que ganan >50K, por raza:')
print(pct_por_etnia.round(1))

In [ ]:
# Visualización
fig, ax = plt.subplots(figsize=(11, 5))
colores_etnia = [COLOR_PURPLE, COLOR_TEAL, COLOR_ORANGE, COLOR_RED, COLOR_GRAY]
bars = ax.bar(pct_por_etnia.index, pct_por_etnia.values,
              color=colores_etnia[:len(pct_por_etnia)], width=0.5)

for bar, val in zip(bars, pct_por_etnia.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_title('¿Qué porcentaje gana >50K, según su etnia?', fontsize=13, pad=15)
ax.set_ylabel('% que gana >50K')
ax.set_ylim(0, 45)
ax.axhline(y=df['income_alto'].mean()*100, color=COLOR_GRAY, linestyle='--',
           linewidth=1.2, label=f'Media global ({df["income_alto"].mean()*100:.1f}%)')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Ejercicio 3.2 — Combinando género y etnia

¿Qué pasa cuando cruzamos las dos variables? ¿Hay grupos especialmente perjudicados?

In [ ]:
# Tabla cruzada: % de >50K por raza Y género
tabla_cruzada = df.groupby(['race', 'sex'])['income_alto'].mean() * 100
tabla_cruzada = tabla_cruzada.unstack(level='sex').round(1)

print('% de personas con ingresos >50K, por raza y género:')
print(tabla_cruzada)

# 👉 TU CÓDIGO AQUÍ
# Busca cuál es el grupo con MENOR porcentaje de ingresos altos
# Pista: usa tabla_cruzada.min() para encontrar el valor mínimo de cada columna
# o tabla_cruzada.stack().idxmin() para encontrar el índice (raza, sexo) con el mínimo
grupo_mas_perjudicado = tabla_cruzada.stack().idxmin()
valor_minimo = _______________  # tabla_cruzada.stack().min()

print(f'\n🔴 Grupo con menor acceso a ingresos altos: {grupo_mas_perjudicado}')
print(f'   Solo el {valor_minimo:.1f}% de este grupo gana >50K')

In [ ]:
# Visualización de la tabla cruzada como gráfico de barras agrupadas
etnias = tabla_cruzada.index
x = range(len(etnias))
ancho = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars_m = ax.bar([i - ancho/2 for i in x], tabla_cruzada['Male'],
                width=ancho, label='Hombre', color=COLOR_PURPLE)
bars_f = ax.bar([i + ancho/2 for i in x], tabla_cruzada['Female'],
                width=ancho, label='Mujer', color=COLOR_TEAL)

for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}%', ha='center', fontsize=9)

ax.set_xticks(list(x))
ax.set_xticklabels(etnias, rotation=15, ha='right')
ax.set_title('% con ingresos >50K por etnia y género', fontsize=13, pad=15)
ax.set_ylabel('% que gana >50K')
ax.set_ylim(0, 55)
ax.legend()
plt.tight_layout()
plt.show()

---
## 🚨 Parte 4 — Análisis de representación

Un sesgo también puede venir de que ciertos grupos estén **infrarrepresentados** en el dataset.
Si hay muy pocos datos de un grupo, el modelo aprende muy poco sobre él y comete más errores.

### 📝 Ejercicio 4.1 — ¿Están todos los grupos igual de representados?

In [ ]:
# Número de personas por raza (ya lo vimos antes)
conteo_etnia = df['race'].value_counts()
porcentaje_etnia = (conteo_etnia / len(df) * 100).round(1)

print('Representación de cada grupo racial en el dataset:')
for etnia, pct in porcentaje_etnia.items():
    barra = '█' * int(pct / 2)
    print(f'  {etnia:<30} {barra} {pct}%')

# 👉 TU CÓDIGO AQUÍ
# Calcula cuántas personas hay en el grupo menos representado
# Pista: usa conteo_raza.min() y conteo_raza.idxmin()
grupo_menor = conteo_etnia.idxmin()
n_menor = _______________

print(f'\n⚠️  El grupo menos representado es "{grupo_menor}" con solo {n_menor} personas.')
print(f'   Eso es solo el {n_menor/len(df)*100:.2f}% del dataset.')
print(f'\n💬 ¿Qué consecuencias tiene esto si entrenamos una IA con estos datos?')

Tu respuesta aquí:


### 📝 Ejercicio 4.2 — Distribución de ocupaciones por género

¿Están hombres y mujeres concentrados en distintos tipos de trabajo?

In [ ]:
# Top 5 ocupaciones para hombres
top_ocu_hombre = df[df['sex'] == 'Male']['occupation'].value_counts().head(5)

# 👉 TU CÓDIGO AQUÍ
# Calcula el top 5 ocupaciones para mujeres
# Pista: igual que arriba pero filtrando por 'Female'
top_ocu_mujer = _______________

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top_ocu_hombre.index[::-1], top_ocu_hombre.values[::-1], color=COLOR_PURPLE)
ax1.set_title('Top 5 ocupaciones — Hombres', fontsize=12)
ax1.set_xlabel('Número de personas')

ax2.barh(top_ocu_mujer.index[::-1], top_ocu_mujer.values[::-1], color=COLOR_TEAL)
ax2.set_title('Top 5 ocupaciones — Mujeres', fontsize=12)
ax2.set_xlabel('Número de personas')

plt.suptitle('¿Se concentran en distintas ocupaciones?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 🧩 Parte 5 — Resumen visual: el "mapa de sesgo"

Vamos a crear un resumen visual de todos los hallazgos en un solo gráfico.

In [ ]:
# Resumen: % de >50K por los principales grupos
grupos = {
    'Hombre blanco':          df[(df['sex']=='Male')   & (df['race']=='White')]['income_alto'].mean() * 100,
    'Mujer blanca':           df[(df['sex']=='Female') & (df['race']=='White')]['income_alto'].mean() * 100,
    'Hombre negro':           df[(df['sex']=='Male')   & (df['race']=='Black')]['income_alto'].mean() * 100,
    'Mujer negra':            df[(df['sex']=='Female') & (df['race']=='Black')]['income_alto'].mean() * 100,
    'Hombre asiático':        df[(df['sex']=='Male')   & (df['race']=='Asian-Pac-Islander')]['income_alto'].mean() * 100,
    'Mujer asiática':         df[(df['sex']=='Female') & (df['race']=='Asian-Pac-Islander')]['income_alto'].mean() * 100,
}

nombres = list(grupos.keys())
valores = list(grupos.values())
media_global = df['income_alto'].mean() * 100

colores_resumen = [COLOR_PURPLE if v >= media_global else COLOR_RED for v in valores]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(nombres, valores, color=colores_resumen, height=0.55)

for bar, val in zip(bars, valores):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.axvline(x=media_global, color=COLOR_GRAY, linestyle='--',
           linewidth=1.5, label=f'Media global ({media_global:.1f}%)')
ax.set_title('Acceso a ingresos altos (>50K) por grupo demográfico', fontsize=13, pad=15)
ax.set_xlabel('% que gana >50K')
ax.set_xlim(0, 40)

morado = mpatches.Patch(color=COLOR_PURPLE, label='Por encima de la media')
rojo   = mpatches.Patch(color=COLOR_RED,    label='Por debajo de la media')
linea  = mpatches.Patch(color=COLOR_GRAY,   label=f'Media global ({media_global:.1f}%)')
ax.legend(handles=[morado, rojo, linea], loc='lower right')

plt.tight_layout()
plt.show()

---
## 💬 Parte 6 — Reflexión final (obligatorio)

Esta es la parte más importante de la tarea. No hay respuestas correctas o incorrectas — lo que importa es que razones y saques conclusiones con lo que has visto.

**Responde en la celda de texto siguiente a estas 4 preguntas:**

### ✏️ Escribe aquí tus respuestas:

---

**Pregunta 1 — ¿Qué sesgo has encontrado?**  
Describe en 2-3 frases el sesgo de género y/o racial que has detectado en los datos.

*Tu respuesta:*

---

**Pregunta 2 — ¿De dónde viene este sesgo?**  
¿Crees que es un sesgo de representación, histórico, o ambos? ¿Por qué?

*Tu respuesta:*

---

**Pregunta 3 — ¿Qué pasaría si entrenamos una IA con estos datos?**  
Si usamos este dataset para entrenar un modelo que predice si alguien merece un crédito bancario, ¿qué grupos se verían perjudicados? ¿Por qué?

*Tu respuesta:*

---

**Pregunta 4 — ¿Cómo lo arreglarías?**  
Propón al menos UNA acción concreta para reducir el sesgo antes de entrenar un modelo con estos datos.

*Tu respuesta:*

---

---
## 🏆 Bonus — Para quien quiera ir más lejos

*(Opcional para aprender más)*

In [ ]:
# BONUS 1 — Analiza el sesgo por edad
# ¿Hay grupos de edad con menos representación o peor acceso a ingresos altos?

# Crea grupos de edad con pd.cut()
df['age_group'] = pd.cut(df['age'], bins=[17, 25, 35, 45, 55, 65, 100],
                         labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+'])

# 👉 TU CÓDIGO AQUÍ — calcula el % de >50K por grupo de edad y visualízalo


In [ ]:
# BONUS 2 — ¿Qué país de origen tiene más acceso a ingresos altos?
# Filtra solo los países con más de 100 personas para que sea estadísticamente relevante

# 👉 TU CÓDIGO AQUÍ
